# Behavioral SRAM Stability Under Threshold Variation

A reproducible numerical study of cross-coupled sigmoid inverters. This notebook walks through the assumptions, geometry, Monte Carlo population, and supply sensitivity. It shares implementation with `sram_model.py` and `run_analysis.py`.

**Scope:** static behavioral stability; no access devices, transistor I–V equations, PDK, read/write operation, or silicon validation. See `docs/model.md`.

## 1. Define the experiment

Four independent threshold values per sample have mean 0.40 V and standard deviation 0.04 V. The illustrative introductory histogram separately uses 0.03 V. Both are uncalibrated choices. The seed and original MT19937 draw order are preserved. The voltage-transfer gain is fixed at 14 V⁻¹ across supplies.

In [1]:
from pathlib import Path
import json
import numpy as np
from IPython.display import display, Markdown
from sram_model import sample_thresholds, snm_reference, snm_batch, summarize
from run_analysis import generate

config = json.loads(Path("config.json").read_text())
samples = sample_thresholds(config["samples"], config["vth_mean_v"], config["vth_sigma_v"], config["seed"])
print(json.dumps(config, indent=2))
print("Sample matrix:", samples.shape)

{
  "seed": 42,
  "samples": 10000,
  "vth_mean_v": 0.4,
  "vth_sigma_v": 0.04,
  "illustration_sigma_v": 0.03,
  "gain_per_v": 14.0,
  "criterion_v": 0.18,
  "supplies_v": [
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95,
    1.0,
    1.05,
    1.1
  ],
  "diagonal_points": 1001,
  "vtc_points": 4097,
  "batch_size": 256
}
Sample matrix: (10000, 4)


## 2. Check the butterfly geometry

Each boundary comes from an inverter VTC, with the second curve mirrored. The largest diagonal square in each closed lobe gives a margin; the smaller margin controls the pair. The independent reference uses analytic VTC inversion by bisection. Cases with only one intersection receive zero margin.

In [2]:
for label, thresholds in [("Matched", [.4]*4), ("Mismatch", [.31, .48, .47, .32])]:
    margin, detail = snm_reference(thresholds, gain_per_v=config["gain_per_v"])
    fast = snm_batch([thresholds], gain_per_v=config["gain_per_v"])[0]
    print(f"{label}: reference={margin*1000:.6f} mV; batched={fast*1000:.6f} mV")

Matched: reference=245.585075 mV; batched=245.584985 mV
Mismatch: reference=165.548103 mV; batched=165.548036 mV


## 3. Generate the population, sweep, figures, and data

The same threshold draws are reused at every supply. The 180 mV criterion is a chosen analysis threshold. Wilson intervals quantify finite-sample uncertainty conditional on this model. Numerical agreement does not establish physical accuracy.

In [3]:
results = generate()

{
  "n": 10000,
  "mean_snm_mv": 229.14563622060987,
  "std_snm_mv": 12.096209615569496,
  "p01_snm_mv": 193.2839789983292,
  "p50_snm_mv": 231.7717944486316,
  "below_threshold_count": 8,
  "below_threshold_pct": 0.08,
  "ci95_low_pct": 0.04054337198968799,
  "ci95_high_pct": 0.1577950253386633
}


## 4. Interpret supply sensitivity

The original sweep kept the normalized VTC shape fixed, equivalent to gain=14/VDD in physical voltage units. The revised sweep keeps gain=14 V⁻¹, matching the original single-cell definition. These alternatives agree at 1.0 V but differ elsewhere; see `docs/validation.md` and `data/baseline_comparison.csv`. The empirical 1st percentile is not a guaranteed worst case, and zero events does not mean zero probability.

![Butterfly curves](figures/step2_butterfly.png)

![Monte Carlo distribution](figures/step3_mc_snm_dist.png)

![Supply sensitivity](figures/step4_vdd_scaling.png)

In [4]:
lines = ["| VDD (V) | Mean SNM (mV) | Below criterion (%) | 95% interval (%) |", "|---|---|---|---|"]
for r in results["supply_sweep"]:
    lines.append(f"| {r["vdd_v"]:.2f} | {r["mean_snm_mv"]:.2f} | {r["below_threshold_pct"]:.2f} | {r["ci95_low_pct"]:.3f}–{r["ci95_high_pct"]:.3f} |")
display(Markdown("\n".join(lines)))

| VDD (V) | Mean SNM (mV) | Below criterion (%) | 95% interval (%) |
|---|---|---|---|
| 0.70 | 107.35 | 100.00 | 99.962–100.000 |
| 0.75 | 126.80 | 100.00 | 99.962–100.000 |
| 0.80 | 146.65 | 100.00 | 99.962–100.000 |
| 0.85 | 166.85 | 89.30 | 88.679–89.891 |
| 0.90 | 187.36 | 24.49 | 23.657–25.343 |
| 0.95 | 208.13 | 2.86 | 2.551–3.205 |
| 1.00 | 229.15 | 0.08 | 0.041–0.158 |
| 1.05 | 250.37 | 0.00 | 0.000–0.038 |
| 1.10 | 271.79 | 0.00 | 0.000–0.038 |

## 5. Validation and next steps

Run `python -m unittest discover -s tests -v` and `python validate.py`. The independent grid study checks 66 cases at three supplies to a 0.01 mV tolerance. Read `docs/validation.md` for the recorded results and original baseline.

The main next research step is a calibrated transistor-level comparison. Correlated mismatch, device sizing, temperature, and read/write biases remain outside this study.

References: [Seevinck et al. (1987)](https://doi.org/10.1109/JSSC.1987.1052809) for SNM background; [NIST](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm) for Wilson intervals.